# Adaptive-Compute Transformer — Phase 2 Hard-Skip Speed Benchmark (Kaggle/Colab)

Measures real wall-clock inference latency for `HardSkipAdaptiveBlock`'s `mlp_only` mechanism against the dense Phase 1 adaptive model and the static baseline, both eager and `torch.compile`'d.

**Context (see `DECISIONS.md`, 2026-09-08 entries, for the full story):** Phase 2 was originally scoped for `learned_gate` (`adaptive_v1`, the router ablation's quality winner), but its trained gate scores turned out to be broadly distributed rather than sparse, so any hard cutoff discards real signal — real hard-skip on it collapses quality (val_lm nearly doubles). Pivoted to `fixed_topk` (`ablation_topk`, the ablation's runner-up on quality): its excluded tokens were *already* trained with exact zero-masking, so real hard-skip reproduces dense output exactly (val_lm matched to 4 decimals on every evaluated batch, Module 3). This notebook benchmarks *that* combination.

Weights don't affect latency (only architecture/shapes/control-flow do) — every model here is built **fresh**, not loaded from a checkpoint, so this notebook needs no dataset upload at all. Built module by module: Setup → Code → Smoke check → Full benchmark → Results.

Runtime > Change runtime type > GPU (T4) before running (Colab), or attach a GPU accelerator (Kaggle).

## Module 1 — Setup

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — enable a GPU accelerator before running"
print("GPU:", torch.cuda.get_device_name(0))

import random, numpy as np
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("seed set:", SEED)

## Module 2 — Code

In [ ]:
%%writefile config.py
"""Config for the baseline static-depth transformer.

Every training run resolves one of these to a dict and dumps it to
experiments/<run_name>/config.json before training starts, so no result
is ever reported without the exact config + seed that produced it.
"""
from dataclasses import dataclass, asdict
from typing import Optional


@dataclass
class BaselineConfig:
    # architecture
    vocab_size: int = 50257  # gpt2 tiktoken encoding
    block_size: int = 256    # context length
    n_layer: int = 12
    n_head: int = 6
    d_model: int = 384
    d_ff: int = 1536         # 4 * d_model
    dropout: float = 0.1
    bias: bool = True

    # data
    dataset_name: str = "Salesforce/wikitext"
    dataset_config: str = "wikitext-103-raw-v1"

    # optimization
    batch_size: int = 32
    grad_accum_steps: int = 1
    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    warmup_steps: int = 500
    max_steps: int = 20000
    eval_interval: int = 500
    eval_iters: int = 100

    # run control
    seed: int = 1337
    run_name: str = "baseline_v1"
    out_dir: str = "experiments"
    device: str = "cuda"
    dtype: str = "bfloat16"  # falls back to float32 in train.py if unsupported
    compile: bool = True
    wandb_project: str = "adaptive-compute-transformer"
    wandb_mode: str = "offline"  # "online" once an API key is configured

    def to_dict(self):
        return asdict(self)


@dataclass
class AdaptiveConfig(BaselineConfig):
    """Same architecture as BaselineConfig (identical data + compute
    budget, per the project's baseline-vs-adaptive comparison) plus the
    router's own hyperparameters and the compute-budget loss (see
    losses.py). Phase 1 trains with a soft gate — the LM loss alone never
    rewards using fewer layers, so target_avg_layers/budget_weight supply
    the missing pressure. Train separate runs at different
    target_avg_layers values to trace the accuracy-vs-avg-layers curve."""

    # router
    router_type: str = "learned_gate"  # see router.build_router; other types added per ablation module
    router_hidden_dim: Optional[int] = None  # None = single Linear(d_model, 1), the lightest option
    topk_capacity: float = 0.5  # fixed_topk only: exact fraction of tokens selected per sequence per layer
    confidence_threshold: float = 0.4309  # confidence_threshold only: empirically calibrated via
                                           # calibrate_threshold.py against the real architecture on
                                           # real WikiText-103 validation data (untrained model) --
                                           # targets nonzero_frac~=0.5 i.e. avg_depth~=6, matching the
                                           # other two ablation arms. Re-calibrate if real training
                                           # behavior drifts meaningfully from this starting estimate.

    # compute budget
    target_avg_layers: float = 6.0  # midpoint of n_layer=12; sweep this across separate runs for the curve
    budget_weight: float = 0.01     # needs empirical tuning on real data; keep LM loss dominant early on

    # run control
    run_name: str = "adaptive_v1"
    ckpt_interval: Optional[int] = None  # None = checkpoint every eval_interval steps; set independently
                                          # to decouple checkpoint safety from (expensive) eval cadence
                                          # on long runs, so a mid-run Colab/Kaggle disconnect never
                                          # loses more than ckpt_interval steps of progress


In [ ]:
%%writefile router.py
"""Per-token routers for the adaptive-depth model.

Each router scores every token at every layer: has this token's
representation stabilized, or does it still need more depth? All router
variants (learned gate, fixed top-k, confidence threshold) share one
forward(x, block_out) call signature, so AdaptiveBlock never needs to
branch on which router type is plugged in -- routers that don't need
block_out (learned gate, fixed top-k) simply ignore it; only
ConfidenceThresholdRouter uses it.
"""
import torch
import torch.nn as nn


class LearnedGateRouter(nn.Module):
    """Outputs a per-token gate in (0, 1): ~1 means "this token still needs
    this layer", ~0 means "treat as converged, freeze it here". Multiplying
    a block's output by this gate (the soft-training scheme) keeps the
    router differentiable end-to-end via ordinary backprop through the
    residual stream — no REINFORCE / straight-through estimator needed.
    """

    def __init__(self, d_model, hidden_dim=None, bias=True):
        super().__init__()
        if hidden_dim is None:
            self.net = nn.Linear(d_model, 1, bias=bias)
        else:
            self.net = nn.Sequential(
                nn.Linear(d_model, hidden_dim, bias=bias),
                nn.GELU(),
                nn.Linear(hidden_dim, 1, bias=bias),
            )

        self.reset_gate_init()

    def reset_gate_init(self):
        """Zero-inits the last layer so every router starts at the same
        bias-controlled gate value regardless of the (random) hidden
        state it's fed, then biases it toward "continue" (~0.88). At step
        0 the base transformer hasn't learned anything yet, so router
        decisions would otherwise be arbitrary; starting near "always
        continue" avoids randomly freezing tokens before there's any
        signal worth routing on, and lets the router earn its way toward
        skipping tokens as the base representations become meaningful.

        Callable again after the fact — a wrapping model's blanket
        weight-init pass (e.g. `model.apply(_init_weights)`) will
        re-init every nn.Linear it finds, including this one, so it
        needs to be reasserted afterward.
        """
        last_linear = self.net if isinstance(self.net, nn.Linear) else self.net[-1]
        nn.init.zeros_(last_linear.weight)
        if last_linear.bias is not None:
            nn.init.constant_(last_linear.bias, 2.0)  # sigmoid(2.0) ~= 0.88

    def forward(self, x, block_out=None):
        """x: (B, T, d_model) -> gate: (B, T), values in (0, 1). block_out
        is accepted (and ignored) so every router shares one call signature
        in AdaptiveBlock.forward() -- only ConfidenceThresholdRouter
        actually needs the block's output."""
        return torch.sigmoid(self.net(x).squeeze(-1))


class FixedTopKRouter(nn.Module):
    """Mixture-of-Depths-style router: a small learned per-token scorer
    ranks all tokens in each sequence, and exactly the top `capacity`
    fraction (rounded, per sequence) get processed -- their gate is their
    own sigmoid score; everyone else gets gate = 0 exactly. Unlike
    LearnedGateRouter, average depth doesn't need a budget loss at all --
    capacity directly and exactly determines what fraction of tokens are
    selected, by construction, every forward pass.

    Note the selected tokens' gate is their SCORE, not 1 -- selection
    (which tokens rank in the top capacity fraction) isn't differentiable,
    so scaling by the score is what gives the scorer's weights a gradient
    signal at all (only from tokens that got selected in a given batch).
    This means the "capacity fraction selected" (exact, a clean
    FLOPs-comparable number) and "avg_depth" as logged elsewhere in this
    project (gates summed across layers, which for this router is
    magnitude-weighted by the score, not a plain selection count) are two
    different quantities -- both worth logging, not interchangeable.
    """

    def __init__(self, d_model, capacity, hidden_dim=None, bias=True):
        super().__init__()
        assert 0.0 < capacity <= 1.0, f"capacity must be in (0, 1], got {capacity}"
        self.capacity = capacity
        if hidden_dim is None:
            self.net = nn.Linear(d_model, 1, bias=bias)
        else:
            self.net = nn.Sequential(
                nn.Linear(d_model, hidden_dim, bias=bias),
                nn.GELU(),
                nn.Linear(hidden_dim, 1, bias=bias),
            )

    def forward(self, x, block_out=None):
        """x: (B, T, d_model) -> gate: (B, T). Exactly
        max(1, min(T, round(capacity * T))) positions per sequence are
        non-zero (= their own sigmoid score); the rest are exactly 0.
        block_out is accepted (and ignored) for signature parity with
        ConfidenceThresholdRouter -- see LearnedGateRouter.forward."""
        B, T, _ = x.shape
        scores = torch.sigmoid(self.net(x).squeeze(-1))  # (B, T)

        k = max(1, min(T, round(self.capacity * T)))
        topk_idx = scores.topk(k, dim=1).indices  # (B, k)
        mask = torch.zeros_like(scores, dtype=torch.bool)
        mask.scatter_(1, topk_idx, True)

        return scores * mask


class ConfidenceThresholdRouter(nn.Module):
    """CALM-inspired confidence-threshold router, using a stabilization
    proxy rather than the literature's full early-exit vocab-head
    approach: gates on how much a block's own output actually changes a
    token's representation, relative to the token's own current
    magnitude -- ||block(x) - x|| / ||x||. A token whose representation
    barely moves under further processing has effectively converged.

    Relative, not absolute: the residual stream's magnitude grows across
    layers in a pre-LN transformer (nothing re-normalizes it between
    blocks), so a fixed absolute delta-norm threshold would be
    miscalibrated by depth alone -- early layers rarely tripping it, late
    layers almost always tripping it, regardless of token difficulty.
    Dividing by ||x|| makes the threshold comparable at every layer.

    No learned parameters -- nothing here is trained by gradient descent
    (there's no budget loss for this router either); average depth is
    controlled purely by the fixed threshold, calibrated empirically.
    The gate is a hard 0/1 (not continuous like the other two routers):
    there's no gradient-training reason to keep it soft, since there's no
    router weight that soft-ness would need to be differentiable for --
    gradient to the underlying block's own weights (attention/MLP) still
    flows normally through the constant gate multiplier either way.
    """

    def __init__(self, threshold, eps=1e-6):
        super().__init__()
        self.threshold = threshold
        self.eps = eps

    def forward(self, x, block_out):
        """x, block_out: (B, T, d_model) -> gate: (B, T), values in
        {0.0, 1.0} exactly."""
        delta_norm = (block_out - x).norm(dim=-1)
        relative_delta = delta_norm / (x.norm(dim=-1) + self.eps)
        return (relative_delta > self.threshold).float()


def build_router(cfg):
    """Constructs the router named by cfg.router_type. Each router class
    keeps its own primitive-argument constructor (decoupled, independently
    testable in isolation) -- this factory is the one place that knows how
    to pull the right fields off cfg for each type, so adding a new router
    type (fixed top-k, confidence threshold, ...) never requires touching
    AdaptiveBlock/AdaptiveTransformer.
    """
    if cfg.router_type == "learned_gate":
        return LearnedGateRouter(cfg.d_model, hidden_dim=cfg.router_hidden_dim)
    if cfg.router_type == "fixed_topk":
        return FixedTopKRouter(cfg.d_model, capacity=cfg.topk_capacity, hidden_dim=cfg.router_hidden_dim)
    if cfg.router_type == "confidence_threshold":
        return ConfidenceThresholdRouter(threshold=cfg.confidence_threshold)
    raise ValueError(f"unknown router_type: {cfg.router_type!r}")


In [ ]:
%%writefile model.py
"""Baseline static-depth decoder-only transformer.

Standard GPT-style architecture (pre-LayerNorm, causal self-attention),
deliberately unremarkable: this is the control model. Later, the
adaptive-depth model will subclass / wrap this same block structure so
that any measured difference is attributable to routing, not to the
base architecture.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from router import build_router


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.d_model = cfg.d_model
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=cfg.bias)
        self.attn_dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.d_model, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.attn_dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
        self.act = nn.GELU()

    def forward(self, x):
        return self.dropout(self.proj(self.act(self.fc(x))))


class Block(nn.Module):
    """One transformer block. Kept as a standalone module (rather than
    inlined in the stack) because the adaptive-depth model will need to
    call blocks individually, per-token, between router decisions."""

    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class BaselineTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # weight tying
        self.tok_emb.weight = self.head.weight

        self.apply(self._init_weights)
        # scaled init for residual projections, per GPT-2 paper
        for name, p in self.named_parameters():
            if name.endswith("proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, (
            f"sequence length {T} exceeds block_size {self.cfg.block_size}"
        )
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )
        return logits, loss

    def num_params(self, exclude_embeddings=True):
        n = sum(p.numel() for p in self.parameters())
        if exclude_embeddings:
            n -= self.pos_emb.weight.numel()
        return n

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


class AdaptiveBlock(nn.Module):
    """One transformer block wrapped with a swappable per-token router
    (see router.build_router). The router produces a gate in (0, 1) from
    the block's input and/or output (which inputs it actually needs
    depends on the router type -- see router.py); the block's contribution
    to the residual stream is blended in by that gate: gate ~= 1 applies
    the block in full, gate ~= 0 leaves the token's state unchanged (full
    attention+MLP skip, not just MLP).

    Phase 1 (soft/dense training): every token still runs through the
    full block every layer — nothing is actually skipped compute-wise
    yet, only down-weighted or masked — regardless of router type. Real
    hard-skip / token-compaction for actual compute savings is a
    separate, later phase.
    """

    def __init__(self, cfg):
        super().__init__()
        self.block = Block(cfg)
        self.router = build_router(cfg)

    def forward(self, x):
        out = self.block(x)
        gate = self.router(x, out)  # (B, T); most routers only use x, ConfidenceThresholdRouter needs out too
        g = gate.unsqueeze(-1)
        x = g * out + (1 - g) * x
        return x, gate


class AdaptiveTransformer(nn.Module):
    """Adaptive-depth counterpart to BaselineTransformer: identical
    architecture (same embeddings, ln_f, head, weight tying, init scheme)
    so any measured difference is attributable to routing — but each
    block is an AdaptiveBlock instead of a plain Block, and forward()
    additionally returns the per-layer, per-token gates so a training
    loop can log average depth used and (later) add a compute-budget
    loss term.
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([AdaptiveBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # weight tying
        self.tok_emb.weight = self.head.weight

        self.apply(self._init_weights)
        # scaled init for residual projections, per GPT-2 paper
        for name, p in self.named_parameters():
            if name.endswith("proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))
        # self.apply() above re-initializes every nn.Linear it finds,
        # including each router's -- reassert the router's own zero-init
        # bootstrap (see LearnedGateRouter.reset_gate_init) afterward.
        # Duck-typed: not every router type has (or needs) a bootstrap
        # reset -- e.g. a non-learned confidence-threshold router has no
        # params to reassert -- so only call it where it actually exists.
        for block in self.blocks:
            if hasattr(block.router, "reset_gate_init"):
                block.router.reset_gate_init()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, (
            f"sequence length {T} exceeds block_size {self.cfg.block_size}"
        )
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))

        gates = []
        for block in self.blocks:
            x, gate = block(x)
            gates.append(gate)
        gates = torch.stack(gates, dim=-1)  # (B, T, n_layer)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )

        # Sum of gates across layers approximates each token's "effective
        # depth" under the soft blend (gate ~= 1 per layer contributes
        # ~1 full layer of compute); averaged over the batch, this is the
        # single number the accuracy-vs-avg-layers curve is built from.
        avg_depth = gates.sum(dim=-1).mean()
        aux = {"gates": gates, "avg_depth": avg_depth}
        return logits, loss, aux

    def num_params(self, exclude_embeddings=True):
        n = sum(p.numel() for p in self.parameters())
        if exclude_embeddings:
            n -= self.pos_emb.weight.numel()
        return n


In [ ]:
%%writefile hard_skip.py
"""Phase 2: real hard-skip inference for a trained learned_gate AdaptiveTransformer.

Phase 1's AdaptiveBlock computes every token through every layer, only
blending the result by a soft gate -- zero wall-clock savings. This module
wraps an already-trained AdaptiveBlock (unmodified; its weights are reused
directly, not copied or retrained -- see DECISIONS.md 2026-09-08) with a
real compute-skipping forward path.

Two mechanisms, both built the same way (gather -> compute -> blend ->
scatter on explicit indices, independent of router logic, wrapped by an
inference-time class that derives those indices from the router's own
per-token score):
  - `hard_skip_block_forward` / `HardSkipAdaptiveBlock` ("full_block"):
    skips attn+MLP together for excluded tokens -- matches the real
    Mixture-of-Depths mechanism (excluded tokens contribute no K/V at
    that layer at all). Found to catastrophically break quality on this
    project's real checkpoint (val_lm nearly doubling at the real
    capacity) -- Phase 1 training never excludes anyone from attention,
    so the model was never taught to route around missing K/V. Kept
    available, no longer the default -- see DECISIONS.md 2026-09-08 addendum.
  - `hard_skip_mlp_only_forward` / `HardSkipMLPOnlyAdaptiveBlock`
    ("mlp_only", the default): keeps attention fully dense (identical to
    Phase 1 in every way) and skips real compute only for the MLP
    sub-layer, only for excluded tokens -- carries none of the above
    mismatch, at the cost of a smaller (but still real) compute saving.

`HardSkipAdaptiveTransformer` wires either mechanism (via `mode=`) into a
full model, mirroring AdaptiveTransformer.forward()'s structure exactly.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def topk_indices_in_order(scores, k):
    """Per-sequence top-k indices by score, re-sorted back into ascending
    (original sequence) order. Critical: gathering in score-rank order
    instead would corrupt the causal mask applied to the compacted
    subsequence -- a later-position token could land before an
    earlier-position one in the gathered tensor.

    scores: (B, T) -> idx: (B, k), each row strictly increasing.
    """
    idx = scores.topk(k, dim=1).indices  # (B, k), score-descending order
    idx, _ = idx.sort(dim=1)
    return idx


def hard_skip_block_forward(block, x, active_idx, scores):
    """x: (B, T, D). active_idx: (B, k) token indices, ascending per row --
    exactly the tokens that get real compute this layer. scores: (B, T),
    the full per-token gate/score (inactive positions' scores are present
    but unused) -- only the active subset's own scores are used to blend
    the active subset's own block output, the same way Phase 1's soft
    blend does.

    Excluded positions are returned bit-for-bit unchanged (gate 0, no
    compute, and no contribution to this layer's attention for anyone
    else either -- they're never part of the gathered subsequence).
    Active positions get block(x)'s real output blended by their own
    score -- identical math to Phase 1's AdaptiveBlock.forward, just
    computed only on the gathered subset.

    Returns (x_new, gate) where gate: (B, T) is 0 at excluded positions
    and the active tokens' own score elsewhere -- same shape/meaning as
    AdaptiveBlock.forward's gate output, so it aggregates into avg_depth
    the same way.
    """
    B, T, D = x.shape
    idx_expanded = active_idx.unsqueeze(-1).expand(-1, -1, D)  # (B, k, D)
    x_active = torch.gather(x, 1, idx_expanded)  # (B, k, D)

    out_active = block(x_active)  # real compute, only on the gathered subset

    scores_active = torch.gather(scores, 1, active_idx)  # (B, k)
    g = scores_active.unsqueeze(-1)
    blended_active = g * out_active + (1 - g) * x_active

    x_new = x.clone()
    x_new.scatter_(1, idx_expanded, blended_active)

    gate = torch.zeros(B, T, dtype=scores.dtype, device=scores.device)
    gate.scatter_(1, active_idx, scores_active)

    return x_new, gate


def hard_skip_mlp_only_forward(block, x, active_idx, scores):
    """Real hard-skip variant that keeps self-attention fully DENSE --
    every token computes and is visible as Q/K/V at every layer, exactly
    matching how Phase 1 training computed attention -- and skips real
    compute only for the MLP sub-layer, only for excluded tokens.

    Added after `hard_skip_block_forward` (full attn+MLP skip) was found,
    on the real adaptive_v1 checkpoint, to catastrophically break quality
    (val_lm nearly doubling even with only ~50% of tokens excluded) --
    see DECISIONS.md 2026-09-08 addendum. Root cause: Phase 1 training
    never excludes any token from attention, so a model trained that way
    was never taught to route around a token vanishing from the K/V pool.
    This variant never changes attention at all, so it carries none of
    that mismatch.

    Uses the algebraic identity that Phase 1's blend formula
    (gate*block(x) + (1-gate)*x, where block(x) = x + attn_delta +
    mlp_delta, since Block is a standard residual block) distributes
    additively: gate*(x + attn_delta + mlp_delta) + (1-gate)*x
    = x + gate*attn_delta + gate*mlp_delta. So scaling each sub-layer's
    own delta by the SAME gate and adding both to x reproduces Phase 1's
    original formula exactly -- this function just never skips computing
    attn_delta (for anyone), and only skips computing mlp_delta for
    excluded tokens (treating their mlp contribution as exactly 0, the
    same approximation `hard_skip_block_forward` already made for the
    whole block, now scoped to just the MLP half).

    Important: mlp_delta must be computed from the UNSCALED post-attention
    residual (x + attn_delta, matching what dense Block.forward actually
    feeds into ln2/mlp) -- NOT from the gate-scaled x + gate*attn_delta.
    Gate scaling is applied once, at the very end, to attn_delta and
    mlp_delta separately. (An earlier draft scaled attn_delta before
    computing mlp_delta from it -- wrong, and caught by the capacity=1.0
    exact-reproduction smoke test, which requires this to be bit-for-bit
    correct rather than merely close.)

    block: an AdaptiveBlock's own `.block` (needs .ln1, .attn, .ln2, .mlp
    -- see model.Block). x: (B, T, D). active_idx: (B, k), ascending per
    row. scores: (B, T), the full per-token score (gate).

    Returns (x_new, gate) -- same shape/meaning as hard_skip_block_forward.
    """
    B, T, D = x.shape
    g_full = scores.unsqueeze(-1)

    # Attention sub-layer: dense, every token, every layer -- identical
    # compute to Phase 1/dense AdaptiveBlock in every way.
    attn_delta = block.attn(block.ln1(x))
    x_plus_attn = x + attn_delta  # UNSCALED -- the correct mlp/ln2 input

    # MLP sub-layer: real compute only for the active subset, fed the
    # same unscaled residual dense Block.forward would use.
    idx_expanded = active_idx.unsqueeze(-1).expand(-1, -1, D)
    x_plus_attn_active = torch.gather(x_plus_attn, 1, idx_expanded)
    mlp_delta_active = block.mlp(block.ln2(x_plus_attn_active))

    scores_active = torch.gather(scores, 1, active_idx)
    g_active = scores_active.unsqueeze(-1)

    x_new = x + g_full * attn_delta  # everyone gets their gate-scaled attn contribution
    mlp_contrib = torch.zeros_like(x)
    mlp_contrib.scatter_(1, idx_expanded, g_active * mlp_delta_active)
    x_new = x_new + mlp_contrib  # excluded tokens' mlp contribution stays exactly 0

    gate = torch.zeros(B, T, dtype=scores.dtype, device=scores.device)
    gate.scatter_(1, active_idx, scores_active)

    return x_new, gate


class HardSkipMLPOnlyAdaptiveBlock(nn.Module):
    """Inference-time hard-skip wrapper using the MLP-only mechanism (see
    hard_skip_mlp_only_forward) -- reuses an already-trained
    AdaptiveBlock's weights directly, no new parameters. Same capacity=1.0
    exact-reproduction invariant as HardSkipAdaptiveBlock."""

    def __init__(self, adaptive_block, capacity):
        super().__init__()
        assert 0.0 < capacity <= 1.0, f"capacity must be in (0, 1], got {capacity}"
        self.block = adaptive_block.block
        self.router = adaptive_block.router
        self.capacity = capacity

    def forward(self, x):
        B, T, D = x.shape
        scores = self.router(x)
        k = max(1, min(T, round(self.capacity * T)))
        active_idx = topk_indices_in_order(scores, k)
        return hard_skip_mlp_only_forward(self.block, x, active_idx, scores)


class HardSkipAdaptiveBlock(nn.Module):
    """Inference-time hard-skip wrapper around an already-trained
    AdaptiveBlock. Creates no parameters of its own -- reuses the given
    block's `.block` (attn+MLP) and `.router` directly, so this is purely
    a different forward path over the same trained weights.

    capacity=1.0 (select every token, every layer) must reproduce the
    wrapped AdaptiveBlock's own dense forward() output exactly -- see the
    Module 2 smoke test. That invariant is what makes this a real
    optimization of the trained model, not a differently-behaving one.
    """

    def __init__(self, adaptive_block, capacity):
        super().__init__()
        assert 0.0 < capacity <= 1.0, f"capacity must be in (0, 1], got {capacity}"
        self.block = adaptive_block.block
        self.router = adaptive_block.router
        self.capacity = capacity

    def forward(self, x):
        B, T, D = x.shape
        scores = self.router(x)  # (B, T) -- needs only x; block_out isn't computed yet
        k = max(1, min(T, round(self.capacity * T)))
        active_idx = topk_indices_in_order(scores, k)
        return hard_skip_block_forward(self.block, x, active_idx, scores)


HARD_SKIP_MODES = {
    "full_block": HardSkipAdaptiveBlock,   # skip attn+MLP together; see DECISIONS.md -- breaks quality on a
                                            # model trained without attention exclusion (Phase 1 soft blend)
    "mlp_only": HardSkipMLPOnlyAdaptiveBlock,  # attention stays dense; only MLP is skipped -- see above
}


class HardSkipAdaptiveTransformer(nn.Module):
    """Inference-time hard-skip wrapper around an already-trained, already-
    loaded AdaptiveTransformer. Reuses every weight (token/position
    embeddings, each block's attn+MLP+router, ln_f, head) directly --
    creates no new parameters, mirrors AdaptiveTransformer.forward()'s
    structure and aux dict exactly so it's a drop-in replacement for
    evaluation/benchmarking code that already expects that shape.

    mode selects which per-layer mechanism to use -- see HARD_SKIP_MODES.
    Default is "mlp_only": the full_block mechanism was found to
    catastrophically break quality on a Phase-1-trained model (see
    DECISIONS.md 2026-09-08 addendum), so it's no longer the default,
    though it's kept available (e.g. for a future fine-tuning experiment
    that trains a model to expect real attention exclusion).
    """

    def __init__(self, adaptive_transformer, capacity, mode="mlp_only"):
        super().__init__()
        if mode not in HARD_SKIP_MODES:
            raise ValueError(f"unknown hard-skip mode: {mode!r}, expected one of {list(HARD_SKIP_MODES)}")
        block_cls = HARD_SKIP_MODES[mode]
        self.cfg = adaptive_transformer.cfg
        self.mode = mode
        self.tok_emb = adaptive_transformer.tok_emb
        self.pos_emb = adaptive_transformer.pos_emb
        self.drop = adaptive_transformer.drop
        self.ln_f = adaptive_transformer.ln_f
        self.head = adaptive_transformer.head
        self.hs_blocks = nn.ModuleList([
            block_cls(b, capacity) for b in adaptive_transformer.blocks
        ])

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, (
            f"sequence length {T} exceeds block_size {self.cfg.block_size}"
        )
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))

        gates = []
        for block in self.hs_blocks:
            x, gate = block(x)
            gates.append(gate)
        gates = torch.stack(gates, dim=-1)  # (B, T, n_layer)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )

        avg_depth = gates.sum(dim=-1).mean()
        aux = {"gates": gates, "avg_depth": avg_depth}
        return logits, loss, aux


In [ ]:
%%writefile visualize_routing.py
"""Token-level routing visualization: which tokens a trained adaptive model
actually processes deeply vs. barely touches, on real sentences.

Needs no new training -- AdaptiveTransformer.forward() already returns
per-layer, per-token gates in aux["gates"] on every call. This module just
loads already-trained checkpoints, runs inference, and renders those gates
as colored text, comparing a shallow-budget and a deep-budget checkpoint
side by side on the same sentences.
"""
import os
import json
from dataclasses import fields

import numpy as np
import torch
from matplotlib import colormaps
from matplotlib.colors import to_hex

from config import AdaptiveConfig
from model import AdaptiveTransformer

EXPERIMENTS_DIR = "experiments"
RESULTS_DIR = "results"
COLORMAP = colormaps["YlOrRd"]

HAND_WRITTEN_SENTENCES = [
    "The quick brown fox jumps over the lazy dog.",
    "The mitochondria is the powerhouse of the cell, according to quantum photosynthesis apparatus.",
    "She said that she would probably go to the store if it stopped raining.",
]


def load_run(run_name, experiments_dir=EXPERIMENTS_DIR, device="cpu"):
    """Reconstructs the AdaptiveConfig a run was trained with from its
    saved config.json, builds the model, and loads its checkpoint weights."""
    run_dir = os.path.join(experiments_dir, run_name)
    with open(os.path.join(run_dir, "config.json")) as f:
        raw_cfg = json.load(f)

    # config.json also carries git_commit/resolved_device, added at save
    # time (see train_adaptive.py) -- not AdaptiveConfig fields, so filter
    # to the dataclass's actual field names before reconstructing it.
    valid_fields = {f.name for f in fields(AdaptiveConfig)}
    cfg_kwargs = {k: v for k, v in raw_cfg.items() if k in valid_fields}
    cfg = AdaptiveConfig(**cfg_kwargs)

    model = AdaptiveTransformer(cfg).to(device)
    ckpt = torch.load(os.path.join(run_dir, "ckpt.pt"), map_location=device, weights_only=False)
    state_dict = ckpt["model"]
    # torch.compile-wrapped models (the real Kaggle runs used compile=True)
    # can save state_dict keys prefixed with "_orig_mod." -- strip it if
    # present so it loads onto the plain (uncompiled) model built here.
    if any(k.startswith("_orig_mod.") for k in state_dict):
        state_dict = {k.removeprefix("_orig_mod."): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.eval()
    return model, cfg


def route_text(model, enc, text, device="cpu"):
    """Tokenizes text and returns [(token_text, effective_depth), ...] --
    effective_depth is a token's gates summed across layers (same quantity
    as AdaptiveTransformer's avg_depth, just per-token instead of averaged
    over the batch), continuous in [0, n_layer]."""
    ids = enc.encode_ordinary(text)
    idx = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        _, _, aux = model(idx)
    depths = aux["gates"].sum(dim=-1).squeeze(0)
    tokens = [enc.decode([tid]) for tid in ids]
    return list(zip(tokens, depths.tolist()))


def get_real_snippet(enc, cache_dir=None, start=40000, n_tokens=25):
    """Pulls a short, real WikiText-103 validation snippet, tokenizing (and
    caching) just the validation split -- not the full ~103M-token train
    split prepare_dataset() would also fetch. `start` was chosen by
    inspecting a few offsets and picking one that reads as a clean,
    coherent clause with none of WikiText-103-raw's "@-@" compound-word
    artifacts (which would be awkward to render per-token-colored)."""
    import data as data_module

    cache_dir = cache_dir or data_module.CACHE_DIR
    os.makedirs(cache_dir, exist_ok=True)
    val_path = os.path.join(cache_dir, "val.bin")
    if not os.path.exists(val_path):
        ids = data_module._tokenize_split("validation", "wikitext-103-raw-v1", enc)
        ids.tofile(val_path)

    data = np.memmap(val_path, dtype=np.uint16, mode="r")
    snippet_ids = data[start:start + n_tokens].astype(np.int64).tolist()
    return enc.decode(snippet_ids)


def depth_to_color(depth, max_depth):
    """Maps an effective depth in [0, max_depth] to a CSS hex color via a
    perceptual sequential colormap (light/cheap -> dark red/expensive)."""
    t = max(0.0, min(1.0, depth / max_depth))
    return to_hex(COLORMAP(t))


def render_html(run_results, sentences_by_label, max_depth, out_path):
    """run_results: {run_name: {label: [(token, depth), ...]}}
    Renders one HTML page with one column per run, sharing the SAME
    max_depth color scale across all columns -- the whole point of the
    comparison is that a shallow-budget run's tokens sit visibly lighter
    across the board, which an independently-normalized-per-run scale
    would hide."""
    parts = ["""<!doctype html><html><head><meta charset="utf-8">
<title>Adaptive routing visualization</title>
<style>
body { font-family: -apple-system, Segoe UI, sans-serif; margin: 2rem; background: #fafafa; color: #111; }
h1 { font-size: 1.3rem; }
.columns { display: flex; gap: 2rem; flex-wrap: wrap; }
.column { flex: 1; min-width: 320px; background: #fff; border: 1px solid #ddd; border-radius: 8px; padding: 1rem 1.25rem; }
.column h2 { font-size: 1rem; margin-top: 0; }
.sentence { line-height: 2.6; margin: 1rem 0; font-size: 1.05rem; }
.tok { padding: 0.1em 0.05em; border-radius: 3px; }
.legend { margin: 1.5rem 0; }
.legend-bar { height: 18px; width: 300px; border-radius: 3px;
  background: linear-gradient(to right, __GRADIENT_STOPS__); }
.legend-labels { display: flex; justify-content: space-between; width: 300px; font-size: 0.8rem; color: #555; }
.note { color: #555; font-size: 0.85rem; max-width: 700px; }
</style></head><body>
<h1>Adaptive-Compute Transformer &mdash; token-level routing</h1>
<p class="note">Color = effective depth used by that token (gate values summed across all 12 layers), continuous
0&ndash;__MAXDEPTH__, same scale in every column below. This is Phase 1 <em>soft</em> routing: no token is
literally skipped yet, every token still passes through every layer, but a lighter color means that layer's
contribution was mostly gated out for that token. Hover a token to see its exact depth.</p>
"""]

    gradient_stops = ", ".join(depth_to_color(d, max_depth) for d in np.linspace(0, max_depth, 6))
    parts[0] = parts[0].replace("__GRADIENT_STOPS__", gradient_stops).replace("__MAXDEPTH__", f"{max_depth:g}")

    parts.append('<div class="legend"><div class="legend-bar"></div>'
                  f'<div class="legend-labels"><span>0 layers</span><span>{max_depth:g} layers</span></div></div>')

    parts.append('<div class="columns">')
    for run_name, results in run_results.items():
        parts.append(f'<div class="column"><h2>{run_name}</h2>')
        for label, pairs in results.items():
            parts.append(f'<div class="sentence"><strong>{label}:</strong><br>')
            for token, depth in pairs:
                color = depth_to_color(depth, max_depth)
                safe_token = (token.replace("&", "&amp;").replace("<", "&lt;")
                              .replace(">", "&gt;"))
                parts.append(
                    f'<span class="tok" style="background:{color}" title="depth={depth:.2f}">{safe_token}</span>'
                )
            parts.append("</div>")
        parts.append("</div>")
    parts.append("</div></body></html>")

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("".join(parts))
    return out_path


if __name__ == "__main__":
    import tiktoken

    enc = tiktoken.get_encoding("gpt2")
    run_names = ["adaptive_target2", "adaptive_target9"]

    sentences = {f"Hand-written {i+1}": s for i, s in enumerate(HAND_WRITTEN_SENTENCES)}
    sentences["Real WikiText-103 validation snippet"] = get_real_snippet(enc)

    run_results = {}
    max_depth = 0
    for run_name in run_names:
        model, cfg = load_run(run_name)
        max_depth = max(max_depth, cfg.n_layer)
        print(f"loaded {run_name}: target_avg_layers={cfg.target_avg_layers}")
        run_results[run_name] = {
            label: route_text(model, enc, text) for label, text in sentences.items()
        }

    out_path = os.path.join(RESULTS_DIR, "routing_visualization.html")
    render_html(run_results, sentences, max_depth, out_path)
    print(f"saved {out_path}")


In [ ]:
%%writefile benchmark_speed.py
"""Module 4: real wall-clock inference-latency benchmark.

Compares three variants at ablation_topk's real architecture (12-layer,
d_model=384, fixed_topk router, capacity=0.5):
  - baseline: BaselineTransformer, static full depth, no routing at all.
  - adaptive_dense: an AdaptiveTransformer (fixed_topk router) run
    through its normal Phase 1 forward path -- full dense compute for
    every token, every layer, exactly like baseline, plus routing/gating
    math on top. Expected to be AT LEAST as slow as baseline, not faster
    -- Phase 1 never skips compute. This is itself half of "the gap
    between theoretical and real speedup" CLAUDE.md wants documented:
    Phase 1 adds routing overhead for zero savings.
  - adaptive_hard_skip: the same model wrapped in
    HardSkipAdaptiveTransformer(mode="mlp_only") -- the (router,
    mechanism) pair verified in Module 3 to preserve quality exactly
    (val_lm matched dense to 4 decimals on every evaluated batch). This
    is where a real speedup, if any, should show up.

All three are built FRESH by default (untrained) rather than from
ablation_topk's real checkpoint -- weights don't affect latency, only
architecture/shapes/control-flow do (see below), so this is a legitimate
simplification, not a shortcut that changes what's being measured.

Benchmarks a single full-sequence forward pass (idx only, no targets --
matches real serving "prefill," not scoring against known future tokens;
see DECISIONS.md's benchmark-scope decision for why this isn't
autoregressive decode).

Local (CPU, this machine) run is a feasibility/no-crash check --
`--smoke` uses tiny shapes/iteration counts so it finishes in a couple of
minutes. torch.compile is OFF by default (`--compile` to attempt it) --
Triton/torch.compile support on Windows CPU is unreliable, and CPU timing
numbers aren't the real deliverable anyway. The real numbers come from
Module 5's Colab/Kaggle GPU notebook, running this same script with
--compile and full (non---smoke) shapes/iters.

Weights don't affect latency (only architecture/shapes/control-flow do),
so `--checkpoint <run_name>` is optional, not required: without it (the
default -- and what Module 5's notebook uses), all three variants are
built FRESH at ABLATION_TOPK_CFG (ablation_topk's real architecture and
router settings, just untrained) -- self-contained, no need to
upload/attach ablation_topk's checkpoint to a Kaggle kernel at all. Pass
`--checkpoint ablation_topk` to instead load the real trained weights
(useful locally, where the files already exist, purely as an extra
consistency check -- the numbers should not meaningfully differ).

Path A, Module 6 (2026-09-10) extended this with `run_sweep()` / `--sweep`:
times all three variants at each of several `topk_capacity` values
(default: the same four points already trained in the accuracy sweep --
0.1667, 0.3333, 0.5, 0.75) instead of one fixed capacity, so the real
wall-clock number exists at every point on the accuracy-vs-capacity
curve, not just one. `run()` (the original, single-capacity function)
and its output file (`benchmark_speed_<device>.json`, already holding
the real, reported Session 10 result) are both left completely
unchanged -- `run_sweep()` is new, additive, and writes to a
differently-named file (`benchmark_speed_sweep_<device>.json`) so it can
never silently overwrite that already-reported number.
"""
import argparse
import json
import os
import time

import torch

from config import AdaptiveConfig, BaselineConfig
from model import AdaptiveTransformer, BaselineTransformer
from hard_skip import HardSkipAdaptiveTransformer
from visualize_routing import load_run

RESULTS_DIR = "results"
REAL_SHAPES = ((1, 256), (32, 256))
SMOKE_SHAPES = ((1, 64), (4, 64))

# ablation_topk's real architecture + router config (see experiments/ablation_topk/config.json)
# -- everything except router_type/topk_capacity is just AdaptiveConfig's own defaults.
ABLATION_TOPK_CFG = AdaptiveConfig(router_type="fixed_topk", topk_capacity=0.5)

# The four capacities Path A already trained real fixed_topk checkpoints at
# (Session 12 / DECISIONS.md 2026-09-10) -- run_sweep()'s default, so the
# wall-clock benchmark covers exactly the same points as the accuracy curve
# (plot_topk_curve.py), not an arbitrarily different set.
CAPACITIES = (0.1667, 0.3333, 0.5, 0.75)


def build_variants(device, checkpoint_run_name=None, experiments_dir="experiments", topk_capacity=None):
    """Builds all three variants either fresh (default -- see module
    docstring for why this is legitimate for a latency benchmark) or from
    a real checkpoint's trained weights.

    topk_capacity: overrides the fresh-init config's capacity (ignored --
    and irrelevant -- when checkpoint_run_name is given, since a loaded
    checkpoint's capacity is baked into its own weights; see run_sweep()'s
    docstring for why sweeping a single checkpoint across OTHER capacities
    isn't done). None (the default) preserves run()'s original,
    already-reported single-capacity behavior exactly: falls back to
    ABLATION_TOPK_CFG (capacity 0.5), unchanged from before this module.
    """
    if checkpoint_run_name is not None:
        adaptive_model, cfg = load_run(checkpoint_run_name, experiments_dir=experiments_dir, device=device)
    else:
        cfg = ABLATION_TOPK_CFG if topk_capacity is None else AdaptiveConfig(
            router_type="fixed_topk", topk_capacity=topk_capacity)
        adaptive_model = AdaptiveTransformer(cfg).to(device)
    adaptive_model.eval()

    baseline_fields = {f: getattr(cfg, f) for f in
                        ("vocab_size", "block_size", "n_layer", "n_head", "d_model", "d_ff", "dropout", "bias")}
    baseline_model = BaselineTransformer(BaselineConfig(**baseline_fields)).to(device)
    baseline_model.eval()

    hard_skip_model = HardSkipAdaptiveTransformer(adaptive_model, capacity=cfg.topk_capacity, mode="mlp_only").to(device)
    hard_skip_model.eval()

    return {
        "baseline": baseline_model,
        "adaptive_dense": adaptive_model,
        "adaptive_hard_skip": hard_skip_model,
    }, cfg


@torch.no_grad()
def time_forward(model, x, warmup, iters, device):
    is_cuda = device == "cuda"
    for _ in range(warmup):
        model(x)
        if is_cuda:
            torch.cuda.synchronize()

    times = []
    for _ in range(iters):
        t0 = time.perf_counter()
        model(x)
        if is_cuda:
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    times.sort()
    n = len(times)
    return {"mean_s": sum(times) / n, "median_s": times[n // 2], "min_s": times[0], "max_s": times[-1], "iters": iters}


def _time_variant_shapes(model, vocab_size, shapes, warmup, iters, device, label):
    timings = {}
    for (B, T) in shapes:
        x = torch.randint(0, vocab_size, (B, T), device=device)
        t = time_forward(model, x, warmup, iters, device)
        timings[f"{B}x{T}"] = t
        print(f"[{label}] shape={B}x{T}: median={t['median_s']*1000:.2f}ms  "
              f"mean={t['mean_s']*1000:.2f}ms  (n={iters})")
    return timings


def run(device, warmup, iters, shapes, use_compile, checkpoint_run_name=None):
    variants, cfg = build_variants(device, checkpoint_run_name=checkpoint_run_name)
    results = {"device": device, "warmup": warmup, "iters": iters, "shapes": list(shapes),
               "compile_attempted": use_compile, "checkpoint_run_name": checkpoint_run_name,
               "variants": {}}

    for name, model in variants.items():
        results["variants"][name] = {
            "eager": _time_variant_shapes(model, cfg.vocab_size, shapes, warmup, iters, device, f"{name}][eager")
        }
        if use_compile:
            try:
                compiled = torch.compile(model)
                results["variants"][name]["compiled"] = _time_variant_shapes(
                    compiled, cfg.vocab_size, shapes, warmup, iters, device, f"{name}][compiled")
            except Exception as e:
                print(f"[{name}][compiled] SKIPPED -- torch.compile failed: {e}")
                results["variants"][name]["compiled"] = {"error": str(e)}

    os.makedirs(RESULTS_DIR, exist_ok=True)
    out_path = os.path.join(RESULTS_DIR, f"benchmark_speed_{device}.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nsaved: {out_path}")
    return results


def run_sweep(device, warmup, iters, shapes, use_compile, capacities=CAPACITIES, checkpoint_run_name=None):
    """Path A, Module 6: times all three variants at each capacity in
    `capacities` (default: the four Path A accuracy-curve points), so the
    real wall-clock number exists at every point on the accuracy-vs-
    capacity curve (plot_topk_curve.py), not just the one Module 4/5
    already measured (capacity 0.5, results/benchmark_speed_<device>.json
    -- untouched by this function, see module docstring).

    baseline's and adaptive_dense's forward-pass latency are
    architecturally capacity-independent: dense mode always computes
    every token through every layer regardless of what capacity the
    router is configured to select (FixedTopKRouter's own top-k/masking
    cost doesn't meaningfully scale with k either). Only
    adaptive_hard_skip's compute path actually changes with capacity.
    Rather than assert that and skip re-timing baseline/adaptive_dense at
    every capacity (which would save time but turn an assumption into an
    unverified one), this times all three variants at every capacity
    anyway -- baseline/adaptive_dense should come back ~flat across
    capacities (up to ordinary run-to-run noise), which is then a real,
    checked confirmation of that claim rather than a bare assertion; only
    adaptive_hard_skip should show an actual capacity-dependent trend.
    """
    if checkpoint_run_name is not None:
        # A checkpoint's weights were trained at one specific capacity,
        # baked into the router's own trained scores -- sweeping a single
        # checkpoint's weights across OTHER capacities at inference time
        # is exactly the off-capacity-inference mismatch Path A's Module 1
        # diagnostic already found catastrophically breaks quality (see
        # DECISIONS.md 2026-09-09). That finding was about quality, not
        # latency, but reusing one checkpoint across capacities here would
        # imply a comparison this project has already shown isn't
        # meaningful -- so a --checkpoint run always uses ONLY that
        # checkpoint's own trained capacity, ignoring `capacities`.
        _, cfg = load_run(checkpoint_run_name, experiments_dir="experiments", device="cpu")
        capacities = [cfg.topk_capacity]
        print(f"[run_sweep] --checkpoint given: ignoring the capacities list, using "
              f"{checkpoint_run_name}'s own trained topk_capacity={cfg.topk_capacity} only")

    results = {"device": device, "warmup": warmup, "iters": iters, "shapes": list(shapes),
               "compile_attempted": use_compile, "checkpoint_run_name": checkpoint_run_name,
               "capacities": list(capacities), "variants_by_capacity": {}}

    for capacity in capacities:
        print(f"\n=== capacity={capacity:.4f} ===")
        variants, cfg = build_variants(device, checkpoint_run_name=checkpoint_run_name, topk_capacity=capacity)
        cap_key = f"{capacity:.4f}"
        results["variants_by_capacity"][cap_key] = {}

        for name, model in variants.items():
            results["variants_by_capacity"][cap_key][name] = {
                "eager": _time_variant_shapes(model, cfg.vocab_size, shapes, warmup, iters, device,
                                               f"cap={capacity:.4f}][{name}][eager")
            }
            if use_compile:
                try:
                    compiled = torch.compile(model)
                    results["variants_by_capacity"][cap_key][name]["compiled"] = _time_variant_shapes(
                        compiled, cfg.vocab_size, shapes, warmup, iters, device,
                        f"cap={capacity:.4f}][{name}][compiled")
                except Exception as e:
                    print(f"[cap={capacity:.4f}][{name}][compiled] SKIPPED -- torch.compile failed: {e}")
                    results["variants_by_capacity"][cap_key][name]["compiled"] = {"error": str(e)}

    os.makedirs(RESULTS_DIR, exist_ok=True)
    out_path = os.path.join(RESULTS_DIR, f"benchmark_speed_sweep_{device}.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nsaved: {out_path}")
    return results


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--warmup", type=int, default=3)
    parser.add_argument("--iters", type=int, default=10)
    parser.add_argument("--compile", action="store_true")
    parser.add_argument("--smoke", action="store_true", help="tiny shapes/iters for a fast local no-crash check")
    parser.add_argument("--checkpoint", default=None,
                         help="load real trained weights from this run (e.g. ablation_topk) instead of a fresh init")
    parser.add_argument("--sweep", action="store_true",
                         help="Path A Module 6: time all variants across multiple topk_capacity values "
                              "(default: 0.1667,0.3333,0.5,0.75) instead of a single default (0.5). "
                              "Writes to a separate output file -- never touches the original single-run result.")
    parser.add_argument("--capacities", default=None,
                         help="comma-separated topk_capacity values for --sweep (default: the Path A sweep)")
    args = parser.parse_args()

    if args.sweep:
        capacities = CAPACITIES if args.capacities is None else tuple(float(c) for c in args.capacities.split(","))
        if args.smoke:
            run_sweep(device=args.device, warmup=1, iters=2, shapes=SMOKE_SHAPES, use_compile=args.compile,
                      capacities=capacities, checkpoint_run_name=args.checkpoint)
        else:
            run_sweep(device=args.device, warmup=args.warmup, iters=args.iters, shapes=REAL_SHAPES,
                      use_compile=args.compile, capacities=capacities, checkpoint_run_name=args.checkpoint)
    elif args.smoke:
        run(device=args.device, warmup=1, iters=2, shapes=SMOKE_SHAPES, use_compile=args.compile,
            checkpoint_run_name=args.checkpoint)
    else:
        run(device=args.device, warmup=args.warmup, iters=args.iters, shapes=REAL_SHAPES, use_compile=args.compile,
            checkpoint_run_name=args.checkpoint)


In [ ]:
import sys
sys.path.insert(0, '.')
import importlib
import config, router, model, hard_skip, visualize_routing, benchmark_speed
for m in (config, router, model, hard_skip, visualize_routing, benchmark_speed):
    importlib.reload(m)
print('code modules loaded OK:', [m.__name__ for m in (config, router, model, hard_skip, visualize_routing, benchmark_speed)])

## Module 3 — Smoke check

Fast correctness+feasibility check on the real GPU before the full grid — tiny shapes, few iterations, `torch.compile` included since that's the one thing that couldn't be verified locally (no C++ compiler on the dev machine — see `DECISIONS.md` / Module 4's local run). If `torch.compile` fails here too, the full run below will just report the same graceful per-variant failure (`benchmark_speed.py`'s own try/except) rather than crash — not a blocker either way, just something to notice before waiting on the full run.

In [ ]:
from benchmark_speed import run, SMOKE_SHAPES

smoke_results = run(device="cuda", warmup=2, iters=5, shapes=SMOKE_SHAPES, use_compile=True)

## Module 4 — Full benchmark

The real deliverable: full-scale shapes — single-sequence (`1x256`, a low-latency "serving" case) and batch=32 (`32x256`, matching every real training/eval run's `batch_size`/`block_size` — a throughput case) — proper warmup and enough timed iterations for a stable median, both eager and `torch.compile`'d, for all three variants (`baseline`, `adaptive_dense`, `adaptive_hard_skip`).

In [ ]:
from benchmark_speed import run, REAL_SHAPES

full_results = run(device="cuda", warmup=10, iters=30, shapes=REAL_SHAPES, use_compile=True)

## Module 5 — Results

In [ ]:
import json, os

with open("results/benchmark_speed_cuda.json") as f:
    r = json.load(f)

print(f"device={r['device']}  warmup={r['warmup']}  iters={r['iters']}  shapes={r['shapes']}\n")

def get_median_ms(variant, backend, shape_key):
    entry = r["variants"][variant].get(backend, {})
    if shape_key not in entry or "error" in entry:
        return None
    return entry[shape_key]["median_s"] * 1000

shape_keys = [f"{b}x{t}" for b, t in r["shapes"]]
variants = ["baseline", "adaptive_dense", "adaptive_hard_skip"]
backends = ["eager", "compiled"] if r["compile_attempted"] else ["eager"]

print(f"{'variant':20s} {'backend':10s}" + "".join(f"{sk:>14s}" for sk in shape_keys))
for v in variants:
    for b in backends:
        vals = [get_median_ms(v, b, sk) for sk in shape_keys]
        row = "".join(f"{(f'{val:.2f}ms' if val is not None else 'FAILED'):>14s}" for val in vals)
        print(f"{v:20s} {b:10s}{row}")

In [ ]:
# Speedup ratios -- this is the actual CLAUDE.md deliverable: real, not
# theoretical, speedup. Reported at the throughput shape (32x256, matching
# training/eval batch/block size) for both backends where available.
throughput_key = shape_keys[-1]  # "32x256"

print(f"speedups at shape {throughput_key}:\n")
for backend in backends:
    base = get_median_ms("baseline", backend, throughput_key)
    dense = get_median_ms("adaptive_dense", backend, throughput_key)
    hard_skip_ms = get_median_ms("adaptive_hard_skip", backend, throughput_key)
    if None in (base, dense, hard_skip_ms):
        print(f"[{backend}] incomplete (torch.compile likely failed) -- skipping ratios")
        continue
    print(f"[{backend}] adaptive_dense vs baseline:      {dense/base:.3f}x "
          f"({'slower' if dense > base else 'faster'} -- Phase 1 adds routing overhead, saves nothing)")
    print(f"[{backend}] adaptive_hard_skip vs baseline:    {hard_skip_ms/base:.3f}x")
    print(f"[{backend}] adaptive_hard_skip vs adaptive_dense: {hard_skip_ms/dense:.3f}x "
          f"({'speedup' if hard_skip_ms < dense else 'no speedup'} -- this is the real Phase 2 number)")

if "compiled" in backends:
    base_eager = get_median_ms("baseline", "eager", throughput_key)
    base_compiled = get_median_ms("baseline", "compiled", throughput_key)
    if base_compiled is not None:
        print(f"\ncompile gap (baseline, {throughput_key}): eager={base_eager:.2f}ms compiled={base_compiled:.2f}ms "
              f"({base_eager/base_compiled:.3f}x) -- the theoretical-vs-real gap CLAUDE.md asks to document")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
x = range(len(variants))
width = 0.35

eager_vals = [get_median_ms(v, "eager", throughput_key) or 0 for v in variants]
ax.bar([i - width/2 for i in x], eager_vals, width, label="eager")

if "compiled" in backends:
    compiled_vals = [get_median_ms(v, "compiled", throughput_key) or 0 for v in variants]
    ax.bar([i + width/2 for i in x], compiled_vals, width, label="compiled")

ax.set_xticks(list(x)); ax.set_xticklabels(variants)
ax.set_ylabel("median latency (ms)")
ax.set_title(f"Phase 2 hard-skip wall-clock benchmark — shape {throughput_key}")
ax.legend()
plt.tight_layout()

results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
plt.savefig(f"{results_dir}/benchmark_speed_{throughput_key}.png", dpi=150)
plt.show()

## Module 6 — Capacity sweep smoke check (Path A)

Path A (see `DECISIONS.md`, 2026-09-09/10) needs a real wall-clock number at every point on the accuracy-vs-capacity curve (`plot_topk_curve.py`), not just the single `capacity=0.5` point Modules 3/4 above already measured. `run_sweep()` (added to `benchmark_speed.py` — its own "Module 6", per `DECISIONS.md`; not to be confused with this notebook's module numbering) repeats the same benchmark at each of the four capacities the accuracy sweep was actually trained at (0.1667, 0.3333, 0.5, 0.75), still building every variant **fresh** (no checkpoint upload needed — same reasoning as the original single-capacity benchmark above). This smoke check catches issues cheaply, across all four capacities, before the longer real sweep below.

In [ ]:
from benchmark_speed import run_sweep, SMOKE_SHAPES

sweep_smoke_results = run_sweep(device="cuda", warmup=2, iters=5, shapes=SMOKE_SHAPES, use_compile=True)

## Module 7 — Full capacity sweep (Path A)

The real deliverable for Path A: the same full-scale shapes, warmup, and iteration counts as Module 4's original single-capacity run above, repeated at each of the four capacities. Takes roughly 4x as long as Module 4 (~15-20 minutes on a T4, vs. a few minutes for one capacity) — still well under the multi-hour scale of the training runs. Writes to `results/benchmark_speed_sweep_cuda.json` — a **separate** file from Module 4's `results/benchmark_speed_cuda.json`, so the already-reported single-capacity result is never overwritten.

In [ ]:
from benchmark_speed import run_sweep, REAL_SHAPES

sweep_full_results = run_sweep(device="cuda", warmup=10, iters=30, shapes=REAL_SHAPES, use_compile=True)

## Module 8 — Sweep Results

Per-capacity latency table and speedup ratios, read directly from the real `benchmark_speed_sweep_cuda.json` this run just wrote — nothing estimated. Also checks the consistency claim from `DECISIONS.md`'s Module 6 entry: `baseline`/`adaptive_dense` latency should come back roughly flat across capacities (dense-mode compute doesn't depend on capacity at all), while only `adaptive_hard_skip` should show a real capacity-dependent trend.

The final accuracy-vs-real-speed **unification** — combining this timing data with the real, already-verified `val_lm` numbers from Path A's training sweep (Session 12) — happens locally afterward, once this notebook's results are downloaded, matching how every other cross-run comparison in this project (`plot_curve.py`, `plot_ablation.py`, `plot_topk_curve.py`) is done locally from downloaded artifacts rather than inside the training/benchmark notebook itself.

In [ ]:
import json

with open("results/benchmark_speed_sweep_cuda.json") as f:
    rs = json.load(f)

print(f"device={rs['device']}  warmup={rs['warmup']}  iters={rs['iters']}  shapes={rs['shapes']}  "
      f"capacities={rs['capacities']}\n")

def get_median_ms_sweep(capacity, variant, backend, shape_key):
    cap_key = f"{capacity:.4f}"
    entry = rs["variants_by_capacity"][cap_key][variant].get(backend, {})
    if shape_key not in entry or "error" in entry:
        return None
    return entry[shape_key]["median_s"] * 1000

shape_keys_sweep = [f"{b}x{t}" for b, t in rs["shapes"]]
variants_sweep = ["baseline", "adaptive_dense", "adaptive_hard_skip"]
backends_sweep = ["eager", "compiled"] if rs["compile_attempted"] else ["eager"]
throughput_key_sweep = shape_keys_sweep[-1]  # "32x256"

print(f"latency at shape {throughput_key_sweep}, all capacities:\n")
print(f"{'capacity':>10s} {'variant':20s} {'backend':10s} {'median_ms':>12s}")
for cap in rs["capacities"]:
    for v in variants_sweep:
        for b in backends_sweep:
            val = get_median_ms_sweep(cap, v, b, throughput_key_sweep)
            val_str = f"{val:.2f}ms" if val is not None else "FAILED"
            print(f"{cap:>10.4f} {v:20s} {b:10s} {val_str:>12s}")

In [ ]:
print(f"real hard-skip speedup vs. adaptive_dense, at shape {throughput_key_sweep}, per capacity:\n")
for cap in rs["capacities"]:
    dense_ms = get_median_ms_sweep(cap, "adaptive_dense", "eager", throughput_key_sweep)
    hs_ms = get_median_ms_sweep(cap, "adaptive_hard_skip", "eager", throughput_key_sweep)
    base_ms = get_median_ms_sweep(cap, "baseline", "eager", throughput_key_sweep)
    if None in (dense_ms, hs_ms, base_ms):
        print(f"capacity={cap:.4f}: incomplete -- skipping")
        continue
    print(f"capacity={cap:.4f}: hard_skip={hs_ms:.2f}ms  dense={dense_ms:.2f}ms  baseline={base_ms:.2f}ms  "
          f"speedup_vs_dense={dense_ms/hs_ms:.3f}x  speedup_vs_baseline={base_ms/hs_ms:.3f}x")

# Consistency check flagged in DECISIONS.md 2026-09-10 (Module 6): baseline
# and adaptive_dense latency should be ~flat across capacities (up to
# ordinary noise), since dense-mode compute doesn't depend on capacity at
# all -- only adaptive_hard_skip should show a real capacity-dependent trend.
base_vals = [get_median_ms_sweep(cap, "baseline", "eager", throughput_key_sweep) for cap in rs["capacities"]]
dense_vals = [get_median_ms_sweep(cap, "adaptive_dense", "eager", throughput_key_sweep) for cap in rs["capacities"]]
if all(v is not None for v in base_vals + dense_vals):
    print(f"\nconsistency check -- baseline across capacities: {[f'{v:.2f}ms' for v in base_vals]} "
          f"(range {max(base_vals)-min(base_vals):.2f}ms)")
    print(f"consistency check -- adaptive_dense across capacities: {[f'{v:.2f}ms' for v in dense_vals]} "
          f"(range {max(dense_vals)-min(dense_vals):.2f}ms)")

In [ ]:
import matplotlib.pyplot as plt
import os

fig, ax = plt.subplots(figsize=(7.5, 5))
for v in variants_sweep:
    vals = [get_median_ms_sweep(cap, v, "eager", throughput_key_sweep) for cap in rs["capacities"]]
    ax.plot(rs["capacities"], vals, "o-", label=v)

ax.set_xlabel("topk_capacity")
ax.set_ylabel(f"median latency (ms), shape {throughput_key_sweep}")
ax.set_title("Phase 2 hard-skip wall-clock benchmark — capacity sweep (eager)")
ax.legend()
plt.tight_layout()

results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
plt.savefig(f"{results_dir}/benchmark_speed_sweep_{throughput_key_sweep}.png", dpi=150)
plt.show()

In [ ]:
# Backup, same cell-block-immediately-after-finishing pattern as the
# training notebooks (see DECISIONS.md / CLAUDE.md's Colab checkpoint
# backup rule) -- results here are cheap to regenerate, but there's no
# reason not to apply the same discipline.
import shutil
backup_name = "benchmark_speed_results_backup"
shutil.make_archive(backup_name, "zip", results_dir)
print(f"Backed up {results_dir} -> {backup_name}.zip")

try:
    from google.colab import files
    files.download(f"{backup_name}.zip")
    print("Download triggered.")
except ImportError:
    print("Not running in Colab — skipping download, but the zip is at:",
          os.path.abspath(f"{backup_name}.zip"))